# Data Generation

## Setup 1

We have varying features from 50 to 500 with increments as 50. And we keep the first 25 features as the truth. 

We have 50 datasets for each feature size, then we can take the average with respect to the 50 repeats for the mislabeled classification task. 

The generation have both 1's to be 0's and 0's to be 1's, which is stored in R_noisy matrix, the truth matrix is R. 

The prediction accuracy is analyzed with respect to the 10% test set of R matrix. 

In [ ]:
import os
import gzip
import pickle
import numpy as np

# PATH_TO_EXP = "/Users/sijianfan/projects/BiSSGL/datasets/simulations"
PATH_TO_EXP = "/work/sfan/projects/BiSSGL/datasets/simulations"
PATH_DATA = os.path.join(PATH_TO_EXP, "n_features")
os.makedirs(PATH_DATA, exist_ok=True)

# Data configuration.
n_features = np.arange(50, 501, 50)
print(n_features)

n_repeats = 50

# Total 8 settings
n_samples, n_objects = 800, 1600
n_rank = 25

scale = 0.05
noise = 0.10

from sgimc.utils import make_imc_data

filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

base_seed = 0x0BADCAFE

for i, n in enumerate(n_features):
    for rep in range(n_repeats):
        seed = base_seed + i * 1000 + rep
        random_state = np.random.RandomState(seed)

        X, W_ideal, Y, H_ideal, R_noisy, R = make_imc_data(
            n_samples,
            n,
            n_objects,
            n,
            n_rank,
            scale=(scale, scale),
            noise=scale * noise,
            binarize=True,
            random_state=random_state,
            return_noisy_only=False,
        )

        data_to_save = {
            "feature_id": i,
            "n_features": n,
            "repeat_id": rep,
            "seed": seed,
            "X": X,
            "Y": Y,
            "R": R,
            "R_noisy": R_noisy,
        }

        filename = os.path.join(PATH_DATA, filename_template.format(n, rep))
        with gzip.open(filename, "wb+", 4) as fout:
            pickle.dump(data_to_save, fout)

[ 50 100 150 200 250 300 350 400 450 500]


## Setup 2

Now, we change the setting to hide some positive samples. The way is to add noise to the generated continuous matrix only for those is labeled as 1's in previous. 

The data matrix dimenstion is set as 400 times 600. 

We still keep varying features from 50 to 500 with increments as 50. And we still keep the first 25 features as the truth. 

For each dataset, we use the doCrossValidationByPairwise from DRIMC to do 10-fold cross-validation. So each dataset will have a 10-fold CV result. 

We then do for 50 datasets for each feature size, then we can take the average with respect to the 50 repeats of 10-fold CV the only-postive-flipped task. 

We don't need to keep both R and R_noisy since we just pick one as the truth, say R. The flipped sign happens in the doCrossValidationByPairwise procedure. 

The prediction accuracy is analyzed with respect to the 1-fold out positive test set and all the negative set of R matrix. 

In [ ]:
import numpy as np
from sklearn.utils import check_random_state


def make_imc_data_masked(
    n_1,
    d_1,
    n_2,
    d_2,
    k,
    scale=0.05,
    noise=0,
    random_state=None,
    binarize=False,
    return_noisy_only=True,
    threshold=0.0,
    target_density=None,
):
    """Create a simple IMC problem.

    Binary case ({0, 1} labels):
      * `threshold`      : an entry is positive (1) iff its continuous score
                           >= threshold. Higher threshold -> sparser positives.
      * `target_density` : if given, overrides `threshold` with the empirical
                           quantile of the continuous matrix so exactly this
                           fraction of entries are positive (before noise).
      * `noise`          : corrupts POSITIVE entries only -- Gaussian noise is
                           added to the continuous score of each 1, re-thresholded,
                           so some 1's become 0's. Negatives are never touched.
      * Returns R (clean truth) and R_noise (truth with some 1 -> 0 flips).
    """
    random_state = check_random_state(random_state)

    assert d_1 >= k and d_2 >= k
    if not isinstance(scale, (tuple, list)):
        assert isinstance(scale, float) and scale > 0
        scale = scale, scale

    X_scale, Y_scale = scale
    X = random_state.normal(scale=X_scale, size=(n_1, d_1))
    Y = random_state.normal(scale=Y_scale, size=(n_2, d_2))

    # fixed weights -- first k features are informative
    W, H = np.eye(d_1, k), np.eye(d_2, k)

    R_cont = np.dot(np.dot(X, W), np.dot(Y, H).T)

    if binarize:
        if target_density is not None:
            threshold = np.quantile(R_cont, 1.0 - target_density)

        # clean truth in {0, 1}
        R = (R_cont >= threshold).astype(float)

        # noisy: perturb ONLY the positive entries, then re-threshold
        R_noise = R.copy()
        if noise > 0:
            pos_mask = R_cont >= threshold
            perturbed = R_cont[pos_mask] + random_state.normal(
                scale=noise, size=int(pos_mask.sum())
            )
            R_noise[pos_mask] = (perturbed >= threshold).astype(float)
            # negatives stay 0 by construction
    else:
        R = R_cont
        R_noise = R_cont.copy()
        if noise > 0:
            R_noise += random_state.normal(scale=noise, size=(n_1, n_2))

    if return_noisy_only:
        return X, W, Y, H, R_noise
    else:
        return X, W, Y, H, R_noise, R

In [ ]:
import os
import gzip
import pickle
import numpy as np

PATH_TO_EXP = "/Users/sijianfan/projects/BiSSGL/datasets/simulations"
# PATH_TO_EXP = "/work/sfan/projects/BiSSGL/datasets/simulations"
PATH_DATA = os.path.join(PATH_TO_EXP, "masked_ones")
os.makedirs(PATH_DATA, exist_ok=True)

# Data configuration.
n_features = np.arange(50, 501, 50)
print(n_features)

n_repeats = 10

# Total 8 settings
n_samples, n_objects = 400, 600
n_rank = 25

scale = 0.05
noise = 0.01

from sgimc.utils import make_imc_data

filename_template = "data_feature_{:03d}_rep_{:02d}.gz"

base_seed = 0x0BADCAFE

for i, n in enumerate(n_features):
    for rep in range(n_repeats):
        seed = base_seed + i * 1000 + rep
        random_state = np.random.RandomState(seed)

        X, W_ideal, Y, H_ideal, R_noisy, R = make_imc_data_masked(
            n_samples,
            n,
            n_objects,
            n,
            n_rank,
            scale=(scale, scale),
            noise=scale * noise,
            binarize=True,
            random_state=random_state,
            return_noisy_only=False,
            target_density=0.1,
        )

        data_to_save = {
            "feature_id": i,
            "n_features": n,
            "repeat_id": rep,
            "seed": seed,
            "X": X,
            "Y": Y,
            "R": R,
            "R_noisy": R_noisy,
        }

        filename = os.path.join(PATH_DATA, filename_template.format(n, rep))
        with gzip.open(filename, "wb+", 4) as fout:
            pickle.dump(data_to_save, fout)

[ 50 100 150 200 250 300 350 400 450 500]
